# NEXUS v13와 선택 Rhino 모델 비교

현재 선택된 건축 매스 157개를 측정했다. Rhino 선언 단위는 mm이지만, 48×10 주거동·25×100 청사와 면적 속성에 맞춰 1좌표=1m로 해석했다. 수평 바닥면 합집합을 층별로 합산하고 체적÷높이로 검산했다. 법정 연면적이나 상세 평면의 실별 면적은 아니다. 기존 역사 상세 Mesh, 지붕, 식재와 용도 불명 연결 요소는 합계에서 제외했다.

원본 추출 스크립트 measurement.py와 grouping.py는 Rhino Python 3 환경에서 __rhino_doc__를 제공하여 실행한다. 이 노트북은 저장된 측정 결과를 읽어 비교를 재계산한다. Rhino 형상이나 선택을 수정하지 않는다.

In [ ]:
import json
from pathlib import Path
root = Path.cwd()
measurements = json.loads((root / 'rhino_measurements.json').read_text(encoding='utf-8'))
groups = json.loads((root / 'comparison_groups.json').read_text(encoding='utf-8'))
comparison = json.loads((root / 'facility_comparison.json').read_text(encoding='utf-8'))
model = measurements['global_union_gfa']
target = 105560
print({'model_m2': model, 'v13_m2': target, 'difference_m2': model-target, 'difference_pct': (model/target-1)*100})
assert abs(sum(groups['group_gfa'].values()) - model) < 0.001
assert groups['max_volume_check_error'] < 0.001
assert sum(r['target'] or 0 for r in comparison) == target
assert abs(sum(r['model'] or 0 for r in comparison) - model) < 0.001
for row in comparison:
    print(row)


객체 속성의 연면적은 전체 건물 값이 층마다 반복되어 있으며 수정 전 값도 남아 있다. 세대수·사용인원·실내 프로그램·실제 주차구획은 매스만으로 검증할 수 없다.

v13 북측 주차장동 치수 94×80m는 7,520㎡이므로 표의 건축면적 5,640㎡와 불일치한다. 합표 목표 22,560㎡로 비교했다.

대지 경계 228,022.1811㎡ vs v13 218,478.066㎡. 동측 8번 획지 11,998.2213㎡는 체험관 3,000 + 야외 12,000㎡의 부지 목표 15,000㎡보다 3,001.7787㎡ 부족하다. 레지던시·교류실의 추가 부지는 별도다.

In [ ]:
boundary = next(r['area'] for r in groups['site_rows'] if r['name']=='구역경계')
plot8 = next(r['area'] for r in groups['site_rows'] if r['name'].startswith('8 창작'))
print({'boundary_m2':boundary,'boundary_difference_m2':boundary-218478.066,'experience_plot_m2':plot8,'plot_shortfall_m2':15000-plot8})
print({'v13_parking_footprint_m2':5640, 'v13_box_dimensions_area_m2':94*80})
